# Database Object Setup

This notebook creates the schemas and Delta tables required by the parcel sorting hub pipeline.

It is intended to be run before ingestion and transformation notebooks. All objects use `IF NOT EXISTS` so the notebook can be rerun without dropping existing data.

## Schemas

Creates separate schemas for each pipeline layer:
- `bronze`: raw ingested data
- `silver`: cleaned and validated data
- `quarantine`: rejected records
- `gold`: analytics-ready outputs

In [0]:
CREATE SCHEMA IF NOT EXISTS parcel_sorting_hub.bronze;

In [0]:
CREATE SCHEMA IF NOT EXISTS parcel_sorting_hub.silver;

In [0]:
CREATE SCHEMA IF NOT EXISTS parcel_sorting_hub.quarantine;

In [0]:
CREATE SCHEMA IF NOT EXISTS parcel_sorting_hub.gold;

## Bronze Tables

Bronze tables store raw CSV data from S3 with minimal transformation.

Most source fields are stored as `STRING` to preserve the raw file values. Metadata columns track the source file and ingestion timestamp.

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.bronze.devices (
    device_id     STRING,
    type          STRING,
    zone          STRING,
    status        STRING,
    _source_file  STRING,
    _ingested_at  TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.bronze.exceptions (
    exception_id  STRING,
    parcel_id     STRING,
    device_id     STRING,
    error_code    STRING,
    status        STRING,
    reported_at   STRING,
    resolved_at   STRING,
    employee_id   STRING,
    _source_file  STRING,
    _ingested_at  TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.bronze.intake (
    delivery_id   STRING,
    dock_id       STRING,
    arrival_time  STRING,
    unload_start  STRING,
    unload_end    STRING,
    parcel_count  STRING,
    truck_plate   STRING,
    source_hub_id STRING,
    _source_file  STRING,
    _ingested_at  TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.bronze.loading (
    loading_id      STRING,
    dock_id         STRING,
    start_time      STRING,
    close_time      STRING,
    parcel_count    STRING,
    route_id        STRING,
    employee_id     STRING,
    _source_file    STRING,
    _ingested_at    TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.bronze.parcels (
    parcel_id           STRING,
    status              STRING,
    weight_kg           STRING,
    length_cm           STRING,
    width_cm            STRING,
    height_cm           STRING,
    service_type        STRING,
    received_at         STRING,
    source_hub_id       STRING,
    delivery_id         STRING,
    destination_hub_id  STRING,
    loading_id          STRING,
    _source_file        STRING,
    _ingested_at        TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.bronze.scans (
    scan_id         STRING,
    parcel_id       STRING,
    device_id       STRING,
    scan_type       STRING,
    result          STRING,
    scan_timestamp  STRING,
    dynamic_weight  STRING,
    _source_file    STRING,
    _ingested_at    TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.bronze.sorting (
    event_id      STRING,
    parcel_id     STRING,
    sorter_id     STRING,
    chute_id      STRING,
    entry_time    STRING,
    result        STRING,
    _source_file  STRING,
    _ingested_at  TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.bronze.status_history (
    history_id        STRING,
    parcel_id         STRING,
    status_name       STRING,
    change_timestamp  STRING,
    _source_file      STRING,
    _ingested_at      TIMESTAMP
)
USING DELTA;

## Silver Tables

Silver tables store cleaned, typed, and validated records.

Business keys are marked as `NOT NULL`, timestamps and numeric fields use proper data types, and `_updated_at` tracks when each record was merged into Silver.

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.devices (
    device_id   STRING NOT NULL,
    type        STRING,
    zone        STRING,
    status      STRING,
    _updated_at TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.exceptions (
    exception_id STRING NOT NULL,
    parcel_id    STRING,
    device_id    STRING,
    error_code   STRING,
    status       STRING,
    reported_at  TIMESTAMP,
    resolved_at  TIMESTAMP,
    employee_id  STRING,
    _updated_at  TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.intake (
    delivery_id   STRING NOT NULL,
    dock_id       STRING,
    arrival_time  TIMESTAMP,
    unload_start  TIMESTAMP,
    unload_end    TIMESTAMP,
    parcel_count  INT,
    truck_plate   STRING,
    source_hub_id STRING,
    _updated_at   TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.loading (
    loading_id    STRING NOT NULL,
    dock_id       STRING,
    start_time    TIMESTAMP,
    close_time    TIMESTAMP,
    parcel_count  INT,
    route_id      STRING,
    employee_id   STRING,
    _updated_at   TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.parcels (
    parcel_id           STRING NOT NULL,
    status              STRING,
    weight_kg           DECIMAL(10,2),
    length_cm           DECIMAL(10,2),
    width_cm            DECIMAL(10,2),
    height_cm           DECIMAL(10,2),
    service_type        STRING,
    received_at         TIMESTAMP,
    source_hub_id       STRING,
    delivery_id         STRING,
    destination_hub_id  STRING,
    loading_id          STRING,
    _updated_at         TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.scans (
    scan_id        STRING NOT NULL,
    parcel_id      STRING,
    device_id      STRING,
    scan_type      STRING,
    result         STRING,
    scan_timestamp TIMESTAMP,
    dynamic_weight DECIMAL(10,2),
    _updated_at    TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.sorting (
    event_id    STRING NOT NULL,
    parcel_id   STRING,
    sorter_id   STRING,
    chute_id    STRING,
    entry_time  TIMESTAMP,
    result      STRING,
    _updated_at TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.status_history (
    history_id       STRING NOT NULL,
    parcel_id        STRING,
    status           STRING,
    change_timestamp TIMESTAMP,
    _updated_at      TIMESTAMP
)
USING DELTA;

## Quarantine Tables

Quarantine tables store records rejected during Silver validation.

They preserve the original raw values and include `_rejection_reason` and `_rejected_at` for auditability and debugging.

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.devices (
    device_id          STRING,
    type               STRING,
    zone               STRING,
    status             STRING,
    _source_file       STRING,
    _rejection_reason  STRING,
    _rejected_at       TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.exceptions (
    exception_id       STRING,
    parcel_id          STRING,
    device_id          STRING,
    error_code         STRING,
    status             STRING,
    reported_at        STRING,
    resolved_at        STRING,
    employee_id        STRING,
    _source_file       STRING,
    _rejection_reason  STRING,
    _rejected_at       TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.intake (
    delivery_id        STRING,
    dock_id            STRING,
    arrival_time       STRING,
    unload_start       STRING,
    unload_end         STRING,
    parcel_count       STRING,
    truck_plate        STRING,
    source_hub_id      STRING,
    _source_file       STRING,
    _rejection_reason  STRING,
    _rejected_at       TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.loading (
    loading_id         STRING,
    dock_id            STRING,
    start_time         STRING,
    close_time         STRING,
    parcel_count       STRING,
    route_id           STRING,
    employee_id        STRING,
    _source_file       STRING,
    _rejection_reason  STRING,
    _rejected_at       TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.parcels (
    parcel_id           STRING,
    status              STRING,
    weight_kg           STRING,
    length_cm           STRING,
    width_cm            STRING,
    height_cm           STRING,
    service_type        STRING,
    received_at         STRING,
    source_hub_id       STRING,
    delivery_id         STRING,
    destination_hub_id  STRING,
    loading_id          STRING,
    _source_file        STRING,
    _rejection_reason   STRING,
    _rejected_at        TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.scans (
    scan_id           STRING,
    parcel_id         STRING,
    device_id         STRING,
    scan_type         STRING,
    result            STRING,
    scan_timestamp    STRING,
    dynamic_weight    STRING,
    _source_file      STRING,
    _rejection_reason STRING,
    _rejected_at      TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.sorting (
    event_id          STRING,
    parcel_id         STRING,
    sorter_id         STRING,
    chute_id          STRING,
    entry_time        STRING,
    result            STRING,
    _source_file      STRING,
    _rejection_reason STRING,
    _rejected_at      TIMESTAMP
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.status_history (
    history_id        STRING,
    parcel_id         STRING,
    status            STRING,
    change_timestamp  STRING,
    _source_file      STRING,
    _rejection_reason STRING,
    _rejected_at      TIMESTAMP
)
USING DELTA;

## Gold Tables

Gold tables contain business-ready aggregations and KPIs built from the Silver layer for reporting and analysis.